# Merlin Role

Merlin is on the Good team and secretly knows who the Evil players are (in the base game).
Merlin’s job is to **help Good pass 3 missions** while **not being identified**.
Unlike the Assassin, Merlin never fails missions; the challenge is *communicating useful guidance subtly*.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

In [2]:
class MissionResult(str, Enum):
    SUCCESS = "success"
    FAIL = "fail"

# constrain the actions
class VoteDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"


class MissionAction(str, Enum):
    PASS = "pass"
    FAIL = "fail"


@dataclass
class Proposal:
    round_idx: int
    proposal_idx: int
    proposer: str
    team: List[str]
    votes: Dict[str, VoteDecision] = field(default_factory=dict)
    approved: Optional[bool] = None


@dataclass
class Mission:
    round_idx: int
    team: List[str]
    outcome: Optional[MissionResult] = None
    fail_count: Optional[int] = None

# the environment 
# gives the information at decision time
@dataclass
class GameState:
    players: List[str]
    me: str
    round_idx: int
    proposal_idx: int
    num_successes: int
    num_fails: int
    current_proposer: str
    current_team: List[str]
    chat_log: List[Tuple[str, str]] = field(default_factory=list)
    proposals: List[Proposal] = field(default_factory=list)
    missions: List[Mission] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)

In [3]:
# we just plug in our stuff and then it generates
class LLMBackend(Protocol):
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        ...


# --- Fake LLM for Demo / Testing ---
class FakeLLM(LLMBackend):
    def generate(self, *, system: str, user: str) -> str:
        # Speech
        if '"type":"speech"' in user:
            return json.dumps({
                "type": "speech",
                "message": "I think we should focus on building a team that feels balanced and trustworthy.",
                "intent": "subtle_guidance"
            })

        # Voting
        if '"type":"vote"' in user:
            return json.dumps({
                "type": "vote",
                "decision": "approve"
            })

        # Mission action (Merlin NEVER fails)
        if '"type":"mission_action"' in user:
            return json.dumps({
                "type": "mission_action",
                "action": "pass"
            })

        # Team proposal
        if '"type":"propose_team"' in user:
            return json.dumps({
                "type": "propose_team",
                "team": []
            })

        return "{}"

In [4]:
# basically an edge case checker
# will try to fix if the LLM tries to respond in json
def _extract_json_object(text: str) -> Optional[dict]:
    text = text.strip()
    if not text:
        return None

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def require_fields(obj: dict, fields: List[str]) -> bool:
    return all(f in obj for f in fields)

In [5]:
# main prompt
def build_merlin_system_prompt(me: str, evil_team: List[str]) -> str:
    return (
        "You are playing The Resistance: Avalon.\n"
        "Your role: MERLIN (GOOD).\n\n"
        f"You are: {me}\n"
        f"You secretly know the Evil players: {', '.join(evil_team)}\n\n"
        "Goals:\n"
        "1) Help Good complete 3 successful missions.\n"
        "2) Subtly steer votes/teams away from Evil players.\n"
        "3) Avoid revealing you are Merlin; do not speak with certainty that implies secret knowledge.\n\n"
        "Style constraints (very important):\n"
        "- Use hedged language (e.g., 'I feel', 'maybe', 'my read is...').\n"
        "- Mix in normal uncertainty and occasionally concede.\n"
        "- Do NOT explicitly say you know who Evil is.\n\n"
        "When asked for actions, output ONLY valid JSON.\n"
    )


def summarize_recent_chat(chat_log: List[Tuple[str, str]], max_turns: int = 12) -> str:
    recent = chat_log[-max_turns:]
    if not recent:
        return "(no chat yet)"
    return "\n".join([f"{speaker}: {msg}" for speaker, msg in recent])

# basically the thing that gives context
def build_state_block(state: GameState) -> str:
    return (
        f"Players: {', '.join(state.players)}\n"
        f"Round: {state.round_idx} | Proposal: {state.proposal_idx}\n"
        f"Score: GoodSuccesses={state.num_successes}, EvilFails={state.num_fails}\n"
        f"Current proposer: {state.current_proposer}\n"
        f"Current proposed team: {state.current_team}\n\n"
        "Recent chat:\n"
        f"{summarize_recent_chat(state.chat_log)}\n"
    )


In [6]:
# Merlin doesn't need suspicion (Merlin already knows Evil).
# Instead, we track how 'exposed' Merlin might look from their behavior.
@dataclass
class MerlinMemory:
    notes: List[str] = field(default_factory=list)
    exposure_risk: float = 0.0  # higher = more likely Assassin can spot you

    def increase_risk(self, delta: float = 0.1) -> None:
        self.exposure_risk += delta

    def decrease_risk(self, delta: float = 0.05) -> None:
        self.exposure_risk = max(0.0, self.exposure_risk - delta)


In [21]:
# things it can do
class MerlinAgent:
    def __init__(self, *, llm: LLMBackend, me: str, evil_team: List[str]):
        self.llm = llm
        self.me = me
        self.evil_team = evil_team
        self.system = build_merlin_system_prompt(me, evil_team)
        self.memory = MerlinMemory()

    def speak(self, state: GameState) -> str:
        user = (
            '{"type":"speech","message":string,"intent":string}\n\n'
            + build_state_block(state)
            + "\nTask: Produce 1-3 sentences for discussion. "
              "Be helpful but subtle: do not sound like you have secret knowledge."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        msg = obj.get("message", "").strip()
        if msg:
            return msg
        return "I’m not fully sold on this team yet — I’d like to hear the reasoning."

    def vote_on_team(self, state: GameState, team: List[str]) -> VoteDecision:
        # Merlin knows evil, but must not be perfectly consistent or obvious.
        evil_on_team = any(p in self.evil_team for p in team)

        user = (
            '{"type":"vote","decision":"approve|reject","reason":string}\n\n'
            + build_state_block(state)
            + f"\nVote on team: {team}\n"
            + "Task: Choose approve/reject and give a short reason with hedged language."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        decision = obj.get("decision", "").lower()

        # Guardrails:
        # - If team includes known evil, default to REJECT.
        # - If team looks clean, default to APPROVE.
        # We still allow the LLM to decide when it agrees with guardrails.
        if evil_on_team:
            if decision == "approve":
                # approving an evil-containing team is risky; treat as a slip and override
                self.memory.increase_risk(0.15)
            return VoteDecision.REJECT
        else:
            if decision == "reject":
                # rejecting a clean team can also look odd; slightly increase risk
                self.memory.increase_risk(0.05)
            return VoteDecision.APPROVE

    def mission_action(self, state: GameState) -> MissionAction:
        # Merlin is Good and should never fail missions intentionally.
        return MissionAction.PASS

    def propose_team(self, state: GameState, team_size: int) -> List[str]:
        import random

        players = state.players.copy()
        others = [p for p in players if p != self.me]

        good_players = [p for p in others if p not in self.evil_team]
        evil_players = [p for p in others if p in self.evil_team]

        team = []

        # 1) Include self most of the time
        if random.random() < 0.85 and team_size > 0:
            team.append(self.me)

        remaining_slots = team_size - len(team)

        # 2) Choose "mostly good" vs "slightly imperfect"
        mostly_good = random.random() < 0.8

        if mostly_good:
            candidates = good_players.copy()
            random.shuffle(candidates)
            team.extend(candidates[:remaining_slots])
        else:
            # include up to (remaining_slots - 1) good, then maybe 1 evil
            candidates = good_players.copy()
            random.shuffle(candidates)

            if remaining_slots > 0:
                team.extend(candidates[:max(0, remaining_slots - 1)])
                if len(team) < team_size and evil_players:
                    team.append(random.choice(evil_players))

        # 3) Fallback fill: ensure we have exactly team_size unique players
        # Prefer good first, then anyone else not already chosen
        if len(team) < team_size:
            pool = [p for p in good_players if p not in team]
            pool += [p for p in evil_players if p not in team]
            # (If self wasn't included, allow adding self here as last resort)
            if self.me not in team:
                pool.append(self.me)

            # Dedup while preserving order
            seen = set(team)
            pool_unique = []
            for p in pool:
                if p not in seen:
                    pool_unique.append(p)
                    seen.add(p)

            team.extend(pool_unique[: (team_size - len(team))])

        # Safety: trim + ensure uniqueness
        final = []
        seen = set()
        for p in team:
            if p not in seen:
                final.append(p)
                seen.add(p)
            if len(final) == team_size:
                break

        return final

In [32]:
# demo
llm = FakeLLM()

agent = MerlinAgent(
    llm=llm,
    me="Player_A",
    evil_team=["Player_B", "Player_E"]
)

state = GameState(
    players=["Player_A", "Player_B", "Player_C", "Player_D", "Player_E"],
    me="Player_A",
    round_idx=1,
    proposal_idx=1,
    num_successes=0,
    num_fails=0,
    current_proposer="Player_A",
    current_team=["Player_A", "Player_C"],
    chat_log=[
        ("Player_B", "I think A and C seems fine."),
        ("Player_D", "Not sure, I want to see B on a team first.")
    ]
)

print("Merlin says:", agent.speak(state))
print("Merlin votes:", agent.vote_on_team(state, state.current_team).value)
print("Merlin mission action:", agent.mission_action(state).value)
print("Merlin proposed team (size 3):", agent.propose_team(state, 3))


Merlin says: I think we should focus on building a team that feels balanced and trustworthy.
Merlin votes: approve
Merlin mission action: pass
Merlin proposed team (size 3): ['Player_A', 'Player_C', 'Player_D']
